## Building A Chatbot
 Design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.



In [3]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable
groq_api_key=os.getenv("GROQ_API_KEY")

In [4]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022C088CC470>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022C0A151340>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Shreyash and I am a  AI/ML Engineer")])

AIMessage(content='Hi Shreyash! 👋 Great to meet an AI/ML engineer. How can I help you today? Whether you’re looking to brainstorm ideas, troubleshoot code, dive into a specific algorithm, or explore the latest research, I’ve got you covered. Just let me know what’s on your mind!', additional_kwargs={'reasoning_content': 'The user says "Hi, My name is Shreyash and I am an AI/ML Engineer". We need to respond appropriately. There\'s no explicit instruction. We should greet and maybe ask about what they need. The user says they are an AI/ML engineer. We can ask what they would like to discuss or help with. We can also mention that we can help with AI/ML topics. Let\'s respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 157, 'prompt_tokens': 88, 'total_tokens': 245, 'completion_time': 0.176368013, 'completion_tokens_details': {'reasoning_tokens': 86}, 'prompt_time': 0.009016807, 'prompt_tokens_details': None, 'queue_time': 0.053545896, 'total_time': 0.18538482}, 'mo

In [6]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Shreyash and I am a  AI/ML Engineer"),
        AIMessage(content="Hello Shreyash! It's nice to meet you. \n\nAs a AI/ML Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content='You’re **Shreyash**, and you’re an **AI/ML Engineer**.', additional_kwargs={'reasoning_content': 'User says "Hey What\'s my name and what do I do?" We have prior conversation: user introduced themselves as Shreyash, AI/ML Engineer. So answer: name Shreyash, profession AI/ML Engineer. We should respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 157, 'total_tokens': 235, 'completion_time': 0.080653481, 'completion_tokens_details': {'reasoning_tokens': 51}, 'prompt_time': 0.008897868, 'prompt_tokens_details': None, 'queue_time': 0.047954756, 'total_time': 0.089551349}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3d587a02fb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e6430-7512-7960-a408-68ebdfae8d31-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 78, 'total_tokens': 235, 'output_token_det

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [7]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

c:\Users\Shreyash Bhamare\Desktop\Gen AI\genai\Lib\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [8]:
config={"configurable":{"session_id":"chat1"}}

In [9]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Shreyash and I am a  AI/ML Engineer")],
    config=config
)

In [10]:
response.content

'Hey Shreyash! 👋🏼 Great to meet an AI/ML engineer. How can I help you today? Are you working on a particular project, looking for insights on a model, or just curious about the latest trends in AI? Let me know what’s on your mind!'

In [11]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='Your name is Shreyash.', additional_kwargs={'reasoning_content': 'User asked: "What\'s my name?" We know from earlier that the user is Shreyash. So we answer "Your name is Shreyash."'}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 160, 'total_tokens': 208, 'completion_time': 0.068538059, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.008065459, 'prompt_tokens_details': None, 'queue_time': 0.046219625, 'total_time': 0.076603518}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3d587a02fb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e6430-a227-7d22-a3a9-d7df256be0ed-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 160, 'output_tokens': 48, 'total_tokens': 208, 'output_token_details': {'reasoning': 32}})

In [12]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'I’m not sure what your name is—could you let me know?'

In [13]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

'Nice to meet you, John! How can I help you today?'

In [14]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is John.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [15]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [16]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Shreyash")]})

AIMessage(content='Hello Shreyash! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'User says "Hi My name is Shreyash". We need to respond. Possibly greet them, ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 97, 'total_tokens': 146, 'completion_time': 0.05055155, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.00542907, 'prompt_tokens_details': None, 'queue_time': 0.04796381, 'total_time': 0.05598062}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5c8ca06ea1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e6434-af1c-7ec3-9dd4-6dc05d82264f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 97, 'output_tokens': 49, 'total_tokens': 146, 'output_token_details': {'reasoning': 26}})

In [17]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

c:\Users\Shreyash Bhamare\Desktop\Gen AI\genai\Lib\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [18]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Shreyash")],
    config=config
)

response

AIMessage(content='Hello Shreyash! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'The user says: "Hi My name is Shreyash". They likely want a greeting and maybe ask for something. They didn\'t ask a question. We can respond with a friendly greeting and ask how we can help.'}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 97, 'total_tokens': 165, 'completion_time': 0.074581073, 'completion_tokens_details': {'reasoning_tokens': 45}, 'prompt_time': 0.006528718, 'prompt_tokens_details': None, 'queue_time': 0.046296787, 'total_time': 0.081109791}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e6435-2bf2-7801-bc17-767084a5766d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 97, 'output_tokens': 68, 'total_tokens': 165, 'output_token_details': {'reasoning': 45}})

In [19]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Shreyash.'

In [20]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [21]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Shreyash")],"language":"Marathi"})
response.content

'नमस्कार, श्रीयाश! आपल्याला कशात मदत हवी आहे?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [23]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [24]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Shreyash ")],"language":"Marathi"},
    config=config
)
repsonse.content

'नमस्कार श्रीयाश! आपण कसे आहात? आपल्याला काही विचारायचे किंवा चर्चा करायची आहे का?'

In [25]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Marathi"},
    config=config,
)

In [26]:
response.content

'आपले नाव श्रीयाश आहे.'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [32]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="hi! I'm bob", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs

max_token changed to 45

In [34]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [35]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

'I don’t have any information about your personal tastes, so I can’t say for sure which ice‑cream flavor you like. If you share a bit about the flavors you enjoy (e.g., chocolate, vanilla, fruity, mint, etc.) or what you’re in the mood for, I can suggest something that might suit you!'

In [36]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked: “What’s 2\u202f+\u202f2?”'

In [37]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

c:\Users\Shreyash Bhamare\Desktop\Gen AI\genai\Lib\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [38]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

'I don’t have that information. Could you let me know what you’d like me to call you?'

In [39]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'You haven’t mentioned a math problem yet. If you have one in mind, feel free to share it and I’ll help you work through it!'